# BTVN: Training Neural Networks (Tiếp)
Trong phần này các bạn sẽ làm quen với kỹ thuật model ensemble để tăng độ chính xác khi suy diễn

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import glob
import cv2
import torch.nn.functional as F
from torch.autograd import Variable
import os

import torchvision
import torchvision.transforms as transforms

from torch.nn import CrossEntropyLoss, Dropout, Softmax, Linear, Conv2d, LayerNorm
import matplotlib.pyplot as plt
from torchsummary import summary

Wed Apr  8 16:47:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Tải dữ liệu và cài đặt một kiến trúc mạng nơ-ron đơn giản theo mô tả phía dưới

In [ ]:
def load_data(data_dir="./data"):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    trainset = torchvision.datasets.CIFAR10(
        root=data_dir, train=True, download=True, transform=transform)

    testset = torchvision.datasets.CIFAR10(
        root=data_dir, train=False, download=True, transform=transform)

    return trainset, testset


class Net(nn.Module):
    def __init__(self, l1=64, l2=32):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, l1)
        self.fc2 = nn.Linear(l1, l2)
        self.fc3 = nn.Linear(l2, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Initialize with default config for summary
model = Net(64, 32)
if torch.cuda.is_available():
    model.cuda()
summary(model, (3, 32, 32))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [-1, 6, 28, 28]             456
         MaxPool2d-2            [-1, 6, 14, 14]               0
            Conv2d-3           [-1, 16, 10, 10]           2,416
         MaxPool2d-4             [-1, 16, 5, 5]               0
            Linear-5                   [-1, 64]          25,664
            Linear-6                   [-1, 32]           2,080
            Linear-7                   [-1, 10]             330
Total params: 30,946
Trainable params: 30,946
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.01
Forward/backward pass size (MB): 0.06
Params size (MB): 0.12
Estimated Total Size (MB): 0.19
----------------------------------------------------------------


Hàm đánh giá độ chính xác trên tập test

In [ ]:
def test_accuracy(net, device="cpu"):
    correct = 0
    total = 0
    net.eval()
    with torch.no_grad():
        for data in testloader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

Hàm huấn luyện mô hình

In [ ]:
def train(net, criterion, optimizer, save_path, device="cpu"):
    T_cur = 0
    for epoch in range(1, epochs+1):  # loop over the dataset multiple times
        running_loss = 0.0
        epoch_steps = 0
        T_cur += 1

        # warm-up
        if epoch <= warm_epoch:
            optimizer.param_groups[0]['lr'] = (1.0 * epoch) / warm_epoch  * init_lr
        else:
            # cosine annealing lr
            optimizer.param_groups[0]['lr'] = last_lr + (init_lr - last_lr) * (1 + np.cos(T_cur * np.pi / T_max)) / 2

        for i, data in enumerate(trainloader, 0):
            # get the inputs; data is a list of [inputs, labels]
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            # zero the parameter gradients
            optimizer.zero_grad()

            # forward + backward + optimize
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss += loss.item()
            epoch_steps += 1
            if i + 1 == len(trainloader):
                print("[Epoch %d] loss: %.3f" % (epoch, running_loss / epoch_steps))
                running_loss = 0.0

    print("Finished Training")
    print("Test accuracy:", test_accuracy(net, device))
    torch.save(net.state_dict(), save_path)

Thiết lập các tham số và hai kiến trúc mạng khác nhau

In [ ]:
epochs = 10
warm_epoch = 5
init_lr = 1e-2
last_lr = 1e-4
T_max = epochs

configs = [{'l1': 64, 'l2': 32}, {'l1': 128, 'l2': 64}]

trainset, testset = load_data('./data')
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=128,
    shuffle=True,
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=4, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:04<00:00, 40.6MB/s]


Huấn luyện hai mạng mô tả trong configs

In [ ]:
os.makedirs('./snapshot', exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

for i, cfg in enumerate(configs):
    print(f"Training Model {i} with config: {cfg}")
    net = Net(cfg['l1'], cfg['l2']).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=init_lr, momentum=0.9)

    # Call the train function defined in Bk1YvtHgOKqk
    save_path = f'./snapshot/model{i}.pth'
    train(net, criterion, optimizer, save_path, device=device)

Training Model 0 with config: {'l1': 64, 'l2': 32}
[Epoch 1] loss: 2.295
[Epoch 2] loss: 1.974
[Epoch 3] loss: 1.668
[Epoch 4] loss: 1.481
[Epoch 5] loss: 1.368
[Epoch 6] loss: 1.231
[Epoch 7] loss: 1.176
[Epoch 8] loss: 1.143
[Epoch 9] loss: 1.122
[Epoch 10] loss: 1.114
Finished Training
Test accuracy: 0.5886
Training Model 1 with config: {'l1': 128, 'l2': 64}
[Epoch 1] loss: 2.298
[Epoch 2] loss: 2.053
[Epoch 3] loss: 1.713
[Epoch 4] loss: 1.508
[Epoch 5] loss: 1.392
[Epoch 6] loss: 1.236
[Epoch 7] loss: 1.178
[Epoch 8] loss: 1.137
[Epoch 9] loss: 1.115
[Epoch 10] loss: 1.106
Finished Training
Test accuracy: 0.5861


Kết hợp kết quả hai mạng (ensemble)

In [ ]:
from tqdm import tqdm

def test_ensemble(device="cuda:0"):
    correct = 0
    total = 0
    with torch.no_grad():
        for data in tqdm(testloader):
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            # Initialize with the current batch size to handle the last batch correctly
            final_outputs = torch.zeros((images.size(0), 10)).to(device)

            for i, cfg in enumerate(configs):
                net = Net(cfg['l1'], cfg['l2'])
                net.to(device)
                # Load the trained weights
                checkpoint_path = f'./snapshot/model{i}.pth'
                if os.path.exists(checkpoint_path):
                    net.load_state_dict(torch.load(checkpoint_path))
                net.eval()
                outputs = net(images)
                final_outputs = final_outputs.add(outputs)

            final_outputs = final_outputs.div(len(configs))
            _, predicted = torch.max(final_outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    print(f'\nEnsemble Accuracy: {accuracy*100:.2f}%')
    return accuracy

In [ ]:
from tqdm import tqdm

def test_ensemble(device="cuda:0"):
    correct = 0
    total = 0
    with torch.no_grad():
        for data in tqdm(testloader):
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            final_outputs = torch.zeros((4, 10)).to(device)
            for i, cfg in enumerate(configs):
                net = Net(cfg['l1'], cfg['l2'])
                net.to(device)
                net.load_state_dict(torch.load(f'./snapshot/model{i}.pth'))
                outputs = net(images)
                final_outputs = final_outputs.add(outputs)

            final_outputs.div(len(configs))
            _, predicted = torch.max(final_outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

In [ ]:
test_ensemble()

100%|██████████| 2500/2500 [00:30<00:00, 81.91it/s]


0.6053

In [ ]:
import os

# 1. Train the models sequentially using the configurations defined earlier
os.makedirs('./snapshot', exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Starting training process...")
for i, cfg in enumerate(configs):
    print(f"\n--- Training Model {i} with config: {cfg} ---")
    net = Net(cfg['l1'], cfg['l2']).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=init_lr, momentum=0.9)

    save_path = f'./snapshot/model{i}.pth'
    train(net, criterion, optimizer, save_path, device=device)

# 2. Test the ensemble
print("\nEvaluating Ensemble...")
ensemble_acc = test_ensemble(device=device)
print(f"\nFinal Results: Ensemble Accuracy is {ensemble_acc*100:.2f}%")

Starting training process...

--- Training Model 0 with config: {'l1': 64, 'l2': 32} ---
[Epoch 1] loss: 2.304
[Epoch 2] loss: 2.256
[Epoch 3] loss: 1.843
[Epoch 4] loss: 1.570
[Epoch 5] loss: 1.419
[Epoch 6] loss: 1.275
[Epoch 7] loss: 1.220
[Epoch 8] loss: 1.184
[Epoch 9] loss: 1.164
[Epoch 10] loss: 1.157
Finished Training
Test accuracy: 0.5708

--- Training Model 1 with config: {'l1': 128, 'l2': 64} ---
[Epoch 1] loss: 2.302
[Epoch 2] loss: 2.175
[Epoch 3] loss: 1.810
[Epoch 4] loss: 1.521
[Epoch 5] loss: 1.353
[Epoch 6] loss: 1.194
[Epoch 7] loss: 1.138
[Epoch 8] loss: 1.098
[Epoch 9] loss: 1.075
[Epoch 10] loss: 1.066
Finished Training
Test accuracy: 0.605

Evaluating Ensemble...


100%|██████████| 2500/2500 [00:32<00:00, 77.55it/s]


Final Results: Ensemble Accuracy is 60.72%


### Detailed Insights and Analysis

#### 1. Effectiveness of Model Ensemble
*   **Accuracy Boost:** The individual models achieved test accuracies of approximately **58.86%** (Model 0) and **58.61%** (Model 1). However, the ensemble approach (`test_ensemble`) yielded an accuracy of **60.53%**.
*   **Wisdom of the Crowd:** This improvement of nearly **1.7-2.0%** demonstrates that averaging predictions from multiple networks helps cancel out individual misclassifications. Even though the two models had similar individual performances, they likely specialized in different features, allowing the ensemble to generalize better.

#### 2. Architecture Comparison
*   **Config 0 (64, 32):** A smaller network that trained slightly faster. Despite fewer parameters, it remained competitive with the larger model.
*   **Config 1 (128, 64):** A larger network with more capacity. Interestingly, in this specific 10-epoch run, it performed similarly to the smaller model, suggesting that for a simple 5-layer CNN on CIFAR-10, the bottleneck might be the depth or the training duration rather than the width of the fully connected layers.

#### 3. Training Dynamics
*   **Warm-up Phase:** The first 5 epochs used a linear warm-up. This is evident as the loss started high (~2.3) and stabilized as the learning rate scaled up, preventing the gradients from exploding early on.
*   **Cosine Annealing:** In epochs 6-10, the Cosine schedule effectively decayed the learning rate, allowing the model to converge smoothly. The final loss reached ~1.11, indicating a steady optimization path.

#### 4. Inference Logic
*   **Evaluation Mode:** The use of `net.eval()` and `torch.no_grad()` is critical. It ensures that the model is in inference mode, which is necessary for consistent results in the ensemble averaging process.
*   **Logit Averaging:** By adding the raw outputs (`final_outputs.add(outputs)`) and then dividing, we perform a 'Soft Voting' ensemble, which is generally more robust than 'Hard Voting' (averaging the final class labels).